# Currency forward strip

This notebook builds the market objects exposed by the `sjd` Python module and prices a strip of physically settled European currency forwards.

Python owns the dates. The binding converts `datetime.date` values to QuantLib dates internally.

Build the module before running the notebook (stages `sjd.pyd` into `python/sjd_runtime`):

`cmake --build --preset debug --target sjd_python`

Shut down the notebook kernel before rebuilding so Windows can overwrite `sjd.pyd`.

In [1]:
import math
import sys
from datetime import date
from pathlib import Path

_cwd = Path.cwd().resolve()
for _candidate in [_cwd, *_cwd.parents]:
    _notebooks = _candidate / "python" / "notebooks"
    if (_notebooks / "sjd_setup.py").is_file():
        if str(_notebooks) not in sys.path:
            sys.path.insert(0, str(_notebooks))
        break
    if (_candidate / "sjd_setup.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise RuntimeError("Could not find python/notebooks/sjd_setup.py from " + str(_cwd))

import sjd_setup

sjd = sjd_setup.load_sjd()
print(sjd.__file__)
print([name for name in dir(sjd) if not name.startswith("_")])


E:\Code\sjd\python\sjd_runtime\sjd.pyd
['AsianForward', 'AsianOption', 'Black76Call', 'Black76Put', 'BuySell', 'DailyFixingCurve', 'DiscountCurve', 'EuropeanForward', 'EuropeanOption', 'ForwardCurve', 'ForwardCurveCMA', 'ForwardCurveStochasticFwdVol', 'Margin', 'Portfolio', 'PortfolioB2CEM', 'PortfolioEvolver', 'PutCall', 'RatesEnvironment', 'RegCapCreditParameters', 'RegCapUnderlyingFactors', 'RegulatoryCapital', 'SimpleMargin', 'SingleFactorEconomicScenarioGenerator', 'ValidationIssue', 'ValidationReport', 'ValidationSeverity', 'VolatilityCurve', 'VolatilityCurveCMA', 'reg_cap_factors']


## Market data

The example quote is domestic currency units per unit of foreign currency. Forward rates are generated from flat domestic and foreign continuously compounded rates; discount factors use the domestic rate.

A `VolatilityCurve` is constructed to demonstrate that it is exposed, but `EuropeanForward` does not use volatility.

For the physically settled constructor, maturity equals settlement. The core value is:

`(market forward - strike) * domestic discount factor * volume * buy/sell sign`

In [2]:
anchor = sjd_setup.ANCHOR
curve_dates = [
    anchor,
    date(2026, 10, 16),
    date(2026, 12, 16),
    date(2027, 3, 16),
    date(2027, 9, 16),
]

frs, drs, vrs = sjd_setup.fx_curves(sjd, curve_dates)

print("curves ok:", frs.is_ok(), drs.is_ok(), vrs.is_ok())


curves ok: True True True


In [3]:
environment = sjd.RatesEnvironment(anchor)
environment.set_frs(frs)
environment.set_drs(drs)
environment.set_vrs(vrs)  # Not required by EuropeanForward.

print("environment:", environment.is_ok(), environment.error_messages())

environment: True 


## Define and validate the strip

Each trade supplies its already-adjusted settlement date. Business-day and settlement-date rules therefore remain in the Python-facing layer.

In [4]:
trade_inputs = [
    {
        "settlement": date(2026, 10, 16),
        "strike": 17.40,
        "volume": 1_000_000.0,
        "side": sjd.BuySell.BUY,
    },
    {
        "settlement": date(2026, 12, 16),
        "strike": 17.80,
        "volume": 750_000.0,
        "side": sjd.BuySell.SELL,
    },
    {
        "settlement": date(2027, 3, 16),
        "strike": 18.10,
        "volume": 500_000.0,
        "side": sjd.BuySell.BUY,
    },
]

strip = [
    sjd.EuropeanForward(
        settlement=trade["settlement"],
        strike=trade["strike"],
        volume=trade["volume"],
        buy_sell=trade["side"],
    )
    for trade in trade_inputs
]

for forward in strip:
    forward.validate(environment)
    if not forward.is_ok():
        raise ValueError(forward.error_messages())

In [5]:
rows = []
for trade, forward in zip(trade_inputs, strip):
    settlement = trade["settlement"]
    market_forward = frs.get_forward(settlement)
    discount_factor = drs.get_discount_factor(settlement)
    value = forward.get_value(environment)
    rows.append(
        (
            settlement,
            trade["strike"],
            market_forward,
            discount_factor,
            value,
        )
    )

print(f"{'Settlement':<12} {'Strike':>10} {'Forward':>10} {'DF':>10} {'Value':>15}")
for settlement, strike, market_forward, discount_factor, value in rows:
    print(
        f"{settlement:%Y-%m-%d} {strike:10.4f} {market_forward:10.4f} "
        f"{discount_factor:10.6f} {value:15,.2f}"
    )

strip_value = sum(row[-1] for row in rows)
print(f"\nTotal strip value: {strip_value:,.2f}")

Settlement       Strike    Forward         DF           Value
2026-10-16    17.4000    17.5432   0.994263      142,382.38
2026-12-16    17.8000    17.6314   0.982699      124,276.24
2027-03-16    18.1000    17.7623   0.965883     -163,094.88

Total strip value: 103,563.75


In [6]:
# Check the binding against the C++ valuation formula for every trade.
for trade, forward in zip(trade_inputs, strip):
    settlement = trade["settlement"]
    sign = 1.0 if trade["side"] == sjd.BuySell.BUY else -1.0
    expected = (
        (frs.get_forward(settlement) - trade["strike"])
        * drs.get_discount_factor(settlement)
        * trade["volume"]
        * sign
    )
    assert math.isclose(
        forward.get_value(environment), expected, rel_tol=1e-12
    )

print("All strip values match the C++ forward valuation formula.")

All strip values match the C++ forward valuation formula.
